In [1]:
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)  # 모든 행 출력
pd.set_option("display.max_columns", None)  # 모든 열 출력
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
import os
from itertools import product
from functions import (load_parameters, load_generation_data, load_price_data, generate_randomized_generation,
generate_rt_scenarios, plot_generation_data, plot_randomized_generation, plot_scenarios_for_generator, plot_rt_scenarios, plot_summary)

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
S, R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data)
P_DA, P_PN = load_price_data()

✅ 총 10개 파일을 불러왔습니다: 1201.csv, 137.csv, 281.csv, 397.csv, 401.csv, 430.csv, 514.csv, 524.csv, 775.csv, 89.csv
📊 데이터 Shape: I=10, T=24, S=30
✅ 시뮬레이션 초기화 완료: S=30, Randomness='high', M1=748.00, M2=2490.00


### Risk Neutral

In [2]:
setRN = gp.Model("set")
setRN.setParam("MIPGap", 1e-7)

a = setRN.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="alpha")
bp = setRN.addVars(T, S, vtype=GRB.CONTINUOUS, name="beta_plus")
bm = setRN.addVars(T, S, vtype=GRB.CONTINUOUS, name="beta_minus")

x = setRN.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
ep = setRN.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="e_plus")
em = setRN.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="e_minus")

yp = setRN.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="y_plus")
ym = setRN.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="y_minus")
z = setRN.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = setRN.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="z_charge")
zd = setRN.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="z_discharge")
d = setRN.addVars(I, I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="d")

p1 = setRN.addVars(I, T, S, vtype=GRB.BINARY, name="p1")
p2 = setRN.addVars(I, T, S, vtype=GRB.BINARY, name="p2")
p3 = setRN.addVars(I, T, S, vtype=GRB.BINARY, name="p3")
p4 = setRN.addVars(I, T, S, vtype=GRB.BINARY, name="p4")

setRN.update()

obj = gp.quicksum(P_DA[t] * a[t] for t in range(T)) + gp.quicksum((1 / S) * (P_RT[t, s] * bp[t, s] - P_PN[t] * bm[t, s]) for t in range(T) for s in range(S))

setRN.setObjective(obj, GRB.MAXIMIZE)

for t in range(T):
    setRN.addConstr(a[t] == gp.quicksum(x[i, t] for i in range(I)))
for t, s in product(range(T), range(S)):
    setRN.addConstr(bp[t, s] == gp.quicksum(ep[i, t, s] for i in range(I)))
    setRN.addConstr(bm[t, s] == gp.quicksum(em[i, t, s] for i in range(I)))

for i, t, s in product(range(I), range(T), range(S)):
    setRN.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + zc[i, t, s] - zd[i, t, s])
    setRN.addConstr(yp[i, t, s] <= R[i, t, s])
    setRN.addConstr(zd[i, t, s] <= z[i, t, s])
    setRN.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
    setRN.addConstr(yp[i, t, s] <= M1 * p3[i, t, s])
    setRN.addConstr(ym[i, t, s] <= M1 * (1 - p3[i, t, s]))
    setRN.addConstr(ym[i, t, s] <= M1 * p2[i, t, s])
    setRN.addConstr(zc[i, t, s] <= M1 * (1 - p2[i, t, s]))
    setRN.addConstr(zc[i, t, s] <= M1 * p1[i, t, s])
    setRN.addConstr(zd[i, t, s] <= M1 * (1 - p1[i, t, s]))
    setRN.addConstr(z[i, t, s] <= K[i])
    setRN.addConstr(z[i, t + 1, s] == z[i, t, s] + zc[i, t, s] - zd[i, t, s])
for i, s in product(range(I), range(S)):
    setRN.addConstr(z[i, 0, s] == K0[i])

for i, t, s in product(range(I), range(T), range(S)):
    setRN.addConstr(ep[i, t, s] == yp[i, t, s] - gp.quicksum(d[i, j, t, s] for j in range(I)))
    setRN.addConstr(em[i, t, s] == ym[i, t, s] - gp.quicksum(d[j, i, t, s] for j in range(I)))
    setRN.addConstr(gp.quicksum(ep[i, t, s] for i in range(I)) <= M2 * p4[i, t, s])
    setRN.addConstr(gp.quicksum(em[i, t, s] for i in range(I)) <= M2 * (1 - p4[i, t, s]))
    setRN.addConstr(d[i, i, t, s] == 0)

setRN.optimize()

if setRN.status == GRB.OPTIMAL:
    print(f"Optimal solution found! Objective value: {setRN.objVal}")
else:
    print("No optimal solution found.")
    
x_vals = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_vals = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) 
ym_vals = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_vals  = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
zc_vals = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
zd_vals = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
ep_vals = np.array([[[ep[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) 
em_vals = np.array([[[em[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
d_vals = np.array([[[[d[i, j, t, s].x for s in range(S)] for t in range(T)] for j in range(I)] for i in range(I)])

In [3]:
# header = (
#         f"{'s':>2} {'t':>2} {'i':>2} | "
#         f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
#         f"{'':>11} {' ':>8} {' ':>8}{'d_ijt':>8} {'d_jit':>8} {'':>8} {'':>8}\n"
#         f"{'':>11}{'':>8} {'':>8} {'e+':>8} {'e-':>8}\n"
#         + "-" * 80
#     )
# print(header)

# for s, t, i in product(range(0,3), range(9, 10), range(2,3)):
#         R_val = R[i, t, s]
#         x_val = x_vals[i, t]
#         yp_val = yp_vals[i, t, s]
#         ym_val = ym_vals[i, t, s]
#         zc_val = zc_vals[i, t, s]
#         zd_val = zd_vals[i, t, s]
#         z_val = z_vals[i, t, s]
#         ep_val = ep_vals[i, t, s]
#         em_val = em_vals[i, t, s]
#         d_ijt = sum(d_vals[i, j, t, s] for j in range(I) if j != i)
#         d_jit = sum(d_vals[j, i, t, s] for j in range(I) if j != i)

#         print(
#             f"{s:>2} {t:>2} {i:>2} | "
#             f"{R_val:>8.2f} {x_val:>8.2f} {yp_val:>8.2f} {ym_val:>8.2f} "
#             f"{zc_val:>8.2f} {zd_val:>8.2f} {z_val:>8.2f}"
#         )
#         print(
#             f"{'':>11}{' ':>8} {' ':>8} {d_ijt:>8.2f} {d_jit:>8.2f} {' ':>8} {' ':>8} \n"
#             f"{'':>11}{' ':>8} {' ':>8} {ep_val:>8.2f} {em_val:>8.2f}\n"
#         )

In [4]:
da_profit = sum(P_DA[t] * x[i, t].X for i in range(I) for t in range(T))
rt_profit = sum(P_RT[t, s] * ep[i, t, s].X / S for i in range(I) for t in range(T) for s in range(S))
pn_cost   = sum(P_PN[t] * em[i, t, s].X / S for i in range(I) for t in range(T) for s in range(S))
total_profit = da_profit + rt_profit - pn_cost

print("[SETTLEMENT MODEL]")
print(f"DA Profit      = {da_profit:.2f}")
print(f"RT Profit      = {rt_profit:.2f}")
print(f"Penalty Cost   = {pn_cost:.2f}")
print(f"Total Profit   = {total_profit:.2f}")

In [5]:
# print("\n[SETTLEMENT MODEL] Day-Ahead Commitment (sum over DERs):")
# total_only_commit = 0
# for t in range(T):
#     commit_t = sum(x[i, t].X for i in range(I))
#     total_only_commit += commit_t
#     # print(f"Time {t}: {commit_t:.2f}")
# print(f"TOTAL: {total_only_commit:.2f}")

# plt.figure(figsize=(8, 4))
# plt.bar(range(T), x_vals.sum(axis=0))
# plt.title("Total Day-Ahead Commitment Over Time")
# plt.xlabel("Hour")
# plt.ylabel("Total x")
# plt.ylim(0,1500)
# plt.grid(True)
# plt.show()

# i=0
# s=3
# zc_single = zc_vals[i, :, s]  
# zd_single = zd_vals[i, :, s]
# z_single = z_vals[i, :, s]
# hours = np.arange(len(zc_single))

# plt.figure(figsize=(10, 5))
# # plt.step(hours, zc_single, where='post', label=f"Charge (DER {i}, Scen {s})", color = 'green', linestyle = "--", linewidth = 1.5)
# plt.step(hours, zd_single, where='post', label=f"Discharge (DER {i}, Scen {s})", linestyle = "--", color = 'red')
# # plt.step(hours, z_single, where='post', label=f"SoC (DER {i}, Scen {s})", color='#00821E', linewidth=2)
# plt.title(f"[Only] Battery Charging/Discharging (DER {i}, Scenario {s})")
# plt.xlabel("Hour")
# plt.ylabel("Energy (kWh)")
# plt.ylim(0, K[i].max()+10)
# plt.xticks(hours)
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()

### Risk Averse

In [6]:
setRA = gp.Model("set")
setRA.setParam("MIPGap", 1e-7)

a = setRA.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="alpha")
bp = setRA.addVars(T, S, vtype=GRB.CONTINUOUS, name="beta_plus")
bm = setRA.addVars(T, S, vtype=GRB.CONTINUOUS, name="beta_minus")

x = setRA.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
ep = setRA.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="e_plus")
em = setRA.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="e_minus")

yp = setRA.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="y_plus")
ym = setRA.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="y_minus")
z = setRA.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = setRA.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="z_charge")
zd = setRA.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="z_discharge")
d = setRA.addVars(I, I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="d")

p1 = setRA.addVars(I, T, S, vtype=GRB.BINARY, name="p1")
p2 = setRA.addVars(I, T, S, vtype=GRB.BINARY, name="p2")
p3 = setRA.addVars(I, T, S, vtype=GRB.BINARY, name="p3")
p4 = setRA.addVars(I, T, S, vtype=GRB.BINARY, name="p4")

Lambda = setRA.addVars(I, S, vtype=GRB.CONTINUOUS, name="Lambda")
Delta = setRA.addVars(I, S, vtype=GRB.CONTINUOUS, lb=0, name="Delta")
Lambda_bar = setRA.addVars(S, vtype=GRB.CONTINUOUS, name="Lambda_bar")
LAMBDA = 5

setRA.update()

obj = gp.quicksum(P_DA[t] * a[t] for t in range(T)) + gp.quicksum((1 / S) * (P_RT[t, s] * bp[t, s] - P_PN[t] * bm[t, s]) for t in range(T) for s in range(S))
risk_penalty = gp.quicksum(Delta[i, s] for i in range(I) for s in range(S)) / S
obj_risk_averse = obj - LAMBDA * risk_penalty
setRA.setObjective(obj_risk_averse, GRB.MAXIMIZE)


for t in range(T):
    setRA.addConstr(a[t] == gp.quicksum(x[i, t] for i in range(I)))
for t, s in product(range(T), range(S)):
    setRA.addConstr(bp[t, s] == gp.quicksum(ep[i, t, s] for i in range(I)))
    setRA.addConstr(bm[t, s] == gp.quicksum(em[i, t, s] for i in range(I)))

for i, t, s in product(range(I), range(T), range(S)):
    setRA.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + zc[i, t, s] - zd[i, t, s])
    setRA.addConstr(yp[i, t, s] <= R[i, t, s])
    setRA.addConstr(zd[i, t, s] <= z[i, t, s])
    setRA.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
    setRA.addConstr(yp[i, t, s] <= M1 * p3[i, t, s])
    setRA.addConstr(ym[i, t, s] <= M1 * (1 - p3[i, t, s]))
    setRA.addConstr(ym[i, t, s] <= M1 * p2[i, t, s])
    setRA.addConstr(zc[i, t, s] <= M1 * (1 - p2[i, t, s]))
    setRA.addConstr(zc[i, t, s] <= M1 * p1[i, t, s])
    setRA.addConstr(zd[i, t, s] <= M1 * (1 - p1[i, t, s]))
    setRA.addConstr(z[i, t, s] <= K[i])
    setRA.addConstr(z[i, t + 1, s] == z[i, t, s] + zc[i, t, s] - zd[i, t, s])
for i, s in product(range(I), range(S)):
    setRA.addConstr(z[i, 0, s] == K0[i])

for i, t, s in product(range(I), range(T), range(S)):
    setRA.addConstr(ep[i, t, s] == yp[i, t, s] - gp.quicksum(d[i, j, t, s] for j in range(I)))
    setRA.addConstr(em[i, t, s] == ym[i, t, s] - gp.quicksum(d[j, i, t, s] for j in range(I)))
    setRA.addConstr(gp.quicksum(ep[i, t, s] for i in range(I)) <= M2 * p4[i, t, s])
    setRA.addConstr(gp.quicksum(em[i, t, s] for i in range(I)) <= M2 * (1 - p4[i, t, s]))
    setRA.addConstr(d[i, i, t, s] == 0)
    
for s in range(S):
    setRA.addConstr(Lambda_bar[s] == (1 / I) * gp.quicksum(Lambda[i, s] for i in range(I)))
    for i in range(I):
        # 개별 DER의 기여도 = 내부 거래량 * (기회비용)
        setRA.addConstr(Lambda[i, s] == gp.quicksum(d[i, j, t, s] * (P_PN[t] - P_RT[t, s]) 
                                                    for j in range(I) if j != i for t in range(T)))
        # 절대편차 정의
        setRA.addConstr(Delta[i, s] >= Lambda[i, s] - Lambda_bar[s])
        setRA.addConstr(Delta[i, s] >= Lambda_bar[s] - Lambda[i, s])

setRA.optimize()

if setRA.status == GRB.OPTIMAL:
    print(f"Optimal solution found! Objective value: {setRA.objVal}")
else:
    print("No optimal solution found.")
    
x_vals = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_vals = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) 
ym_vals = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_vals  = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
zc_vals = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
zd_vals = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
ep_vals = np.array([[[ep[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) 
em_vals = np.array([[[em[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
d_vals = np.array([[[[d[i, j, t, s].x for s in range(S)] for t in range(T)] for j in range(I)] for i in range(I)])

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-07
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-07

Optimize a model with 125094 rows, 153834 columns and 616434 nonzeros
Model fingerprint: 0x3e8fcf41
Variable types: 125034 continuous, 28800 integer (28800 binary)
Coefficient statistics:
  Matrix range     [1e-01, 2e+03]
  Objective range  [2e-01, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+03]
Presolve removed 68417 rows and 70655 columns
Presolve time: 0.80s
Presolved: 56677 rows, 83179 columns, 257114 nonzeros
Variable types: 68472 continuous, 14707 integer (14707 binary)
Found heuristic solution: objective 1077981.5565
Deterministic concurrent LP optimizer: primal and dual

In [16]:
# header = (
#         f"{'s':>2} {'t':>2} {'i':>2} | "
#         f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
#         f"{'':>11} {' ':>8} {' ':>8}{'d_ijt':>8} {'d_jit':>8} {'':>8} {'':>8}\n"
#         f"{'':>11}{'':>8} {'':>8} {'e+':>8} {'e-':>8}\n"
#         + "-" * 80
#     )
# print(header)

# for s, t, i in product(range(0,3), range(9, 10), range(2,3)):
#         R_val = R[i, t, s]
#         x_val = x_vals[i, t]
#         yp_val = yp_vals[i, t, s]
#         ym_val = ym_vals[i, t, s]
#         zc_val = zc_vals[i, t, s]
#         zd_val = zd_vals[i, t, s]
#         z_val = z_vals[i, t, s]
#         ep_val = ep_vals[i, t, s]
#         em_val = em_vals[i, t, s]
#         d_ijt = sum(d_vals[i, j, t, s] for j in range(I) if j != i)
#         d_jit = sum(d_vals[j, i, t, s] for j in range(I) if j != i)

#         print(
#             f"{s:>2} {t:>2} {i:>2} | "
#             f"{R_val:>8.2f} {x_val:>8.2f} {yp_val:>8.2f} {ym_val:>8.2f} "
#             f"{zc_val:>8.2f} {zd_val:>8.2f} {z_val:>8.2f}"
#         )
#         print(
#             f"{'':>11}{' ':>8} {' ':>8} {d_ijt:>8.2f} {d_jit:>8.2f} {' ':>8} {' ':>8} \n"
#             f"{'':>11}{' ':>8} {' ':>8} {ep_val:>8.2f} {em_val:>8.2f}\n"
#         )

 s  t  i |        R        x       y+       y-       zc       zd        z
                                d_ijt    d_jit                  
                                   e+       e-
--------------------------------------------------------------------------------
 0  9  2 |    11.00     0.00    11.00     0.00     0.00     0.00     2.00
                                11.00     0.00                   
                                 0.00     0.00

 1  9  2 |    10.00     0.00     0.00     0.00    10.00     0.00     4.00
                                 0.00     0.00                   
                                 0.00     0.00

 2  9  2 |    27.00     0.00    24.00     0.00     3.00     0.00     3.00
                                24.00     0.00                   
                                 0.00     0.00



In [17]:
da_profit = sum(P_DA[t] * x[i, t].X for i in range(I) for t in range(T))
rt_profit = sum(P_RT[t, s] * ep[i, t, s].X / S for i in range(I) for t in range(T) for s in range(S))
pn_cost   = sum(P_PN[t] * em[i, t, s].X / S for i in range(I) for t in range(T) for s in range(S))
total_profit = da_profit + rt_profit - pn_cost

print("[SETTLEMENT MODEL]")
print(f"DA Profit      = {da_profit:.2f}")
print(f"RT Profit      = {rt_profit:.2f}")
print(f"Penalty Cost   = {pn_cost:.2f}")
print(f"Total Profit   = {total_profit:.2f}")

[SETTLEMENT MODEL]
DA Profit      = 258657.79
RT Profit      = 535957.44
Penalty Cost   = 5415.63
Total Profit   = 789199.60


In [7]:
# print("\n[SETTLEMENT MODEL] Day-Ahead Commitment (sum over DERs):")
# total_only_commit = 0
# for t in range(T):
#     commit_t = sum(x[i, t].X for i in range(I))
#     total_only_commit += commit_t
#     # print(f"Time {t}: {commit_t:.2f}")
# print(f"TOTAL: {total_only_commit:.2f}")

# plt.figure(figsize=(8, 4))
# plt.bar(range(T), x_vals.sum(axis=0))
# plt.title("Total Day-Ahead Commitment Over Time")
# plt.xlabel("Hour")
# plt.ylabel("Total x")
# plt.ylim(0,1500)
# plt.grid(True)
# plt.show()

# i=0
# s=3
# zc_single = zc_vals[i, :, s]  
# zd_single = zd_vals[i, :, s]
# z_single = z_vals[i, :, s]
# hours = np.arange(len(zc_single))

# plt.figure(figsize=(10, 5))
# # plt.step(hours, zc_single, where='post', label=f"Charge (DER {i}, Scen {s})", color = 'green', linestyle = "--", linewidth = 1.5)
# plt.step(hours, zd_single, where='post', label=f"Discharge (DER {i}, Scen {s})", linestyle = "--", color = 'red')
# # plt.step(hours, z_single, where='post', label=f"SoC (DER {i}, Scen {s})", color='#00821E', linewidth=2)
# plt.title(f"[Only] Battery Charging/Discharging (DER {i}, Scenario {s})")
# plt.xlabel("Hour")
# plt.ylabel("Energy (kWh)")
# plt.ylim(0, K[i].max()+10)
# plt.xticks(hours)
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()